# Does reordering pay off more where the order is worse?

`reordering-l1-ablation.ipynb` measured L1 at one service window (900 s) and
found it wins on everything: mean ride −12%, max ride −11%, wait −2%, coverage
+0.8 pp, no regression anywhere. It also found *why* — not by freeing slack
for new requests (refusals barely moved, 47.6% → 46.5%) but by giving the
riders already aboard a better visit order.

If that is the mechanism, then L1 should pay most where the order is worst.
And the 800-trial baseline says exactly where that is:

| | No window | W450 | W600 | W750 | W900 |
|---|---|---|---|---|---|
| Served | 82.6 % | 82.8 % | 87.5 % | 91.1 % | 93.1 % |
| Mean ride | **356 s** | 874 s | 710 s | 596 s | 521 s |
| Max ride | 1536 s | **4756 s** | 3802 s | 3154 s | 2591 s |

`W450` carries a mean ride of 874 s and a worst ride of 4756 s — nearly four
times the fixed-route cycle cap. A tight deadline makes the bus chase expiring
pickups, deferring dropoffs and dragging whoever is aboard. That is an
*ordering* pathology, and it is precisely what a repair pass exists to fix.
L1 has never been tried there.

## What is at stake

Two of the project's open questions turn on this.

**Can a demand-responsive policy beat fixed-route outright?** `W750` currently
wins on served, mean wait, P90 wait and mean ride, and loses only on max ride
(3154 s against fixed-route's structural 1302 s). L1 took 11% off the worst
ride at W900. If it does the same at W750 the gap narrows but does not close;
if it does proportionally more where the tail is fatter, the picture changes.

**Which window should we actually recommend?** The current answer is a
frontier — `W900` for coverage, `W750` for the best all-round trade,
no-window for ride quality. Reordering may not shift each column by the same
amount, in which case the recommendation moves.

## The questions, and what would settle them

### Q1 — does the gain scale with how bad the order is?

Plot L1's ride improvement against L0's ride time for the same window. The
mechanism predicts a positive relationship.

- **Improvement grows as the baseline worsens** → confirms the ride-quality
  mechanism and makes L1 most valuable exactly where the policy is weakest.
- **Roughly constant across windows** → the mechanism is real but the ~12%
  measured at W900 is all there is, wherever it is applied.
- **Improvement shrinks where the baseline is worst** → refutes the mechanism.
  A pathological order that repair *cannot* fix would mean the problem is not
  the order but the deadline forcing it, and no amount of L2 would help
  either.

### Q2 — does the refusal rate stay flat?

At W900 it barely moved, which is what pushed the explanation away from the
slack story. Tighter windows refuse far more, so there is more room for a
slack effect to appear if one exists.

- **Flat at every window** → the ride-quality mechanism is the whole story.
- **Falls at tight windows** → the slack channel is real but only bites when
  the window is the binding constraint, which would be worth knowing before
  designing L2.

### Q3 — does L1 change which window we recommend?

Read the frontier before and after. The specific thing to watch is whether any
L1 column reaches fixed-route's 1302 s max ride while keeping ≥ 80% served.
None currently does.

### Q4 — does reordering help when nothing is binding?

The no-window ablation already has the best mean ride on the map (356 s) and
refuses nobody. If L1 still improves it, reordering is a general win rather
than compensation for a badly-behaved window.

In [ ]:
%matplotlib inline
# Static comparison charts only, so the inline backend is all we need.

import statistics
import time
from collections import defaultdict
from typing import Callable

import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import Markdown, display

from src.algorithm.DispatchPolicy import BusState, DispatchPolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.RequestPool import Request
from src.algorithm.Schedule import Schedule
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import (
    TrialResult,
    run_trial,
    summarize_comparison,
)
from src.map.PlanarMapProvider import PlanarMapProvider
from src.map.TransitGraph import TransitGraph
from src.map.graph import PlanarStreetGraph


class _RejectionCounting:
    """
    Mixin recording how often the policy refused to schedule a request.

    Carried over from ``reordering-l1-ablation.ipynb``. The reason it counts
    per call site rather than in total: L1 reaches ``_best_insertion`` from
    three places — first-pass insertion, relocation, and retry — and pooling
    them compares L0's first-pass attempts against a denominator four to five
    times larger for L1. Relocations almost always succeed, so the pooled rate
    collapses and manufactures an improvement that is pure bookkeeping.

    Two measures of refusal are kept: **first-pass rejections** (the
    mechanism) and **requests left unscheduled** (which depends on no internal
    call structure and is therefore the safer headline).

    Counts live on the instance and are read back through ``stats()``. One
    policy is built per trial, so run totals are a sum over instances.
    """

    def __init__(self, *args: object, **kwargs: object) -> None:
        """Forward every argument to the policy being wrapped."""
        super().__init__(*args, **kwargs)
        self._counts: dict[str, float] = {
            "plan_calls": 0.0,
            "offered": 0.0,
            "left_unscheduled": 0.0,
            "insertion_attempts": 0.0,
            "rejections": 0.0,
            "relocation_attempts": 0.0,
        }
        self._context = "insertion"

    def plan(self, state: BusState, n_stops: int) -> list[str]:
        """Plan normally, then record who was left waiting."""
        self._context = "insertion"
        offered = {request.id for request in state.pending_requests}
        route = super().plan(state, n_stops)
        self._counts["plan_calls"] += 1
        self._counts["offered"] += len(offered)
        self._counts["left_unscheduled"] += len(
            offered - self.schedule.owners()
        )
        return route

    def _best_insertion(
        self, schedule: Schedule, request: Request, state: BusState
    ) -> Schedule | None:
        """Insert normally, then tally the outcome against its call site."""
        result = super()._best_insertion(schedule, request, state)
        if self._context == "relocation":
            self._counts["relocation_attempts"] += 1
        else:
            self._counts["insertion_attempts"] += 1
            if result is None:
                self._counts["rejections"] += 1
        return result

    def _relocate_pair(
        self, schedule: Schedule, request: Request, state: BusState
    ) -> Schedule | None:
        """Mark the re-placement so it is not counted as a first insertion."""
        self._context = "relocation"
        try:
            return super()._relocate_pair(schedule, request, state)
        finally:
            self._context = "insertion"

    def stats(self) -> dict[str, float]:
        """
        Merge these counters with any the wrapped policy reports.
        :return: Counter name to value.
        """
        merged = dict(self._counts)
        inherited = getattr(super(), "stats", None)
        if inherited is not None:
            merged.update(inherited())
        return merged


class CountingL0(_RejectionCounting, WindowInsertionPolicy):
    """Insert-only baseline, instrumented."""


class CountingL1(_RejectionCounting, ReorderingInsertionPolicy):
    """Relocate-repair policy, instrumented."""


# Every policy built during the run, per column. Cleared by the run cell.
INSTANCES: dict[str, list[object]] = defaultdict(list)


def make_factory(
    label: str, policy_class: type, window: float,
) -> Callable[[TransitGraph], DispatchPolicy]:
    """
    Build a policy factory that registers what it builds.

    A named closure rather than a lambda, so the late-binding trap that bites
    list comprehensions cannot arise.
    :param label: Column label the instances are filed under.
    :param policy_class: Policy to construct.
    :param window: Service window in seconds.
    :return: Callable taking a TransitGraph and returning a policy.
    """
    def factory(graph: TransitGraph) -> DispatchPolicy:
        policy = policy_class(graph, max_wait_s=window)
        INSTANCES[label].append(policy)
        return policy
    return factory

## 1. Configuration

### Choosing `N_TRIALS`

Set from the margins the previous runs actually produced, rather than by
guesswork. At **200** paired trials the L1 ablation resolved:

| metric | delta | half-width | ratio |
|---|---|---|---|
| Mean ride | −63.7 s | ±6.9 | 9.2 : 1 |
| Mean wait | −9.3 s | ±3.2 | 2.9 : 1 |
| Served | +0.8 pp | ±0.3 | 2.7 : 1 |
| **Max ride** | −291 s | **±108** | **2.7 : 1** |

Mean ride is never in doubt. **Max ride is the constraint**, and it is the
metric Q3 turns on. Half-widths shrink with the square root of the trial
count, so `400` cuts every figure above by ~29% — max ride to about ±76 s —
which is enough to compare improvements *between windows* rather than merely
confirming each is non-zero. Going to 800 would buy another 29% for four times
the runtime of 200, which is not worth it here.

### Runtime

Measured, one trial per column:

| | no window | W450 | W600 | W750 | W900 |
|---|---|---|---|---|---|
| L0 | 1.3 s | 1.6 s | 1.5 s | 1.6 s | 2.0 s |
| L1 | **9.3 s** | 3.2 s | 3.6 s | 5.7 s | 7.0 s |
| ratio | **7.2x** | 2.0x | 2.4x | 3.5x | 3.5x |

That is 36.8 s per trial across all ten columns, so **400 trials is about
4.1 hours**. Halve it by dropping to 200: every headline conclusion survives
except the between-window comparison of max ride, which is the one that needs
the tighter interval.

!!! note "L1's cost scales far worse than L0's"
    The unbounded column costs 7.2x its baseline against 3.5x at W900, because
    with no deadline every request is accepted and the schedule grows. Repair
    is *n* relocations each costing roughly O(n³), so its total is close to
    O(n⁴) in schedule length while plain insertion is O(n³). Worth
    remembering before pointing L1 at a busier map — and another argument for
    filtering which relocations are worth attempting at all.

In [ ]:
N_TRIALS = 400
BASE_SEED = 0

# Stands in for "no window at all": large enough that the deadline never
# binds, isolating what persistent insertion alone builds.
NO_WINDOW = 1e9

# The four swept windows plus the unbounded ablation, matching the columns in
# baseline-c0.ipynb so L0 here can be checked against it.
WINDOWS: list[tuple[str, float]] = [
    ("no window", NO_WINDOW),
    ("W450", 450.0),
    ("W600", 600.0),
    ("W750", 750.0),
    ("W900", 900.0),
]

# run_trial kwargs matching every previous campus experiment. NOTE: max_wait_s
# here is the *backend's abandonment threshold*, a different quantity from a
# policy's service window; commitment is passed explicitly even though 0 is
# now the default, because it is load-bearing for these results.
LOAD = dict(
    max_requests=200, capacity=40, max_wait_s=900.0, commitment=0,
)

LEVELS: list[tuple[str, type]] = [("L0", CountingL0), ("L1", CountingL1)]


def column(window_name: str, level: str) -> str:
    """
    Column label for one (window, level) cell.
    :param window_name: Short window name from WINDOWS.
    :param level: "L0" or "L1".
    :return: The label results and instances are keyed by.
    """
    return f"{window_name} {level}"


POLICIES: list[tuple[str, Callable[[TransitGraph], DispatchPolicy]]] = [
    (column(window_name, level),
     make_factory(column(window_name, level), policy_class, window))
    for window_name, window in WINDOWS
    for level, policy_class in LEVELS
]
COLUMNS = [label for label, _ in POLICIES]
print(f"{len(COLUMNS)} columns: {', '.join(COLUMNS)}")

## 2. Shared map

Identical to every previous sweep — same `MAP_SEED`, same campus
configuration — so every L0 column here is directly comparable to its
counterpart in `baseline-c0.ipynb`.

In [ ]:
MAP_SEED = 42

graph = PlanarStreetGraph.generate(
    n_seeds=17,
    bounds=(0.0, 0.0, 2000.0, 2000.0),
    seed=MAP_SEED,
    stop_density=0.5,
)

provider = PlanarMapProvider(graph)
print(f"{len(provider.get_stops())} bus stops on this map")

## 3. Cost check

L1 measured 3.4× L0 at W900. It should be cheaper at tight windows (fewer
accepted requests means a shorter schedule, and insertion cost grows roughly
with the cube of schedule length) and dearer with no window at all. One trial
per column before committing to four hundred.

In [ ]:
probe_costs: dict[str, float] = {}
for label, factory in POLICIES:
    start = time.perf_counter()
    run_trial(
        policy_factory=factory, graph=graph, seed=BASE_SEED,
        silent=True, **LOAD,
    )
    probe_costs[label] = time.perf_counter() - start

INSTANCES.clear()  # discard the probes' policies

lines = ["| Window | L0 | L1 | ratio |", "| --- | --- | --- | --- |"]
for window_name, _ in WINDOWS:
    l0_cost = probe_costs[column(window_name, "L0")]
    l1_cost = probe_costs[column(window_name, "L1")]
    lines.append(
        f"| {window_name} | {l0_cost:.1f} s | {l1_cost:.1f} s "
        f"| {l1_cost / l0_cost:.1f}x |"
    )
projected_s = N_TRIALS * sum(probe_costs.values())
lines.append("")
lines.append(
    f"Projected total ({N_TRIALS} trials x {len(COLUMNS)} columns): "
    f"**{projected_s / 3600:.1f} hours**"
)
display(Markdown("\n".join(lines)))

## 4. The run

One call, so every column shares `base_seed` and trial *i* faces the same
demand sequence throughout — which is what makes the paired differences below
valid.

In [ ]:
INSTANCES.clear()  # re-running this cell replaces the tallies

summary = summarize_comparison(
    policies=POLICIES,
    graph=graph,
    n_trials=N_TRIALS,
    base_seed=BASE_SEED,
    **LOAD,
)

## 5. What L1 buys at each window (Q1, Q3)

Paired per-trial differences, L1 minus L0, within each window. Positive is
better for served fraction; negative is better for every timing row.

In [ ]:
def paired_delta(
    baseline: list[TrialResult],
    treatment: list[TrialResult],
    extract: Callable[[TrialResult], float | None],
) -> tuple[float, float, float]:
    """
    Mean per-trial difference (treatment - baseline) with a CI and a p-value.

    Trials are paired by index, which is meaningful because every column ran
    in one summarize_comparison call, so trial i faced the same demand
    sequence throughout.
    :param baseline: Per-trial results for the reference column.
    :param treatment: Per-trial results for the column being tested.
    :param extract: Maps a TrialResult to the scalar being compared, or None
        when that trial does not define it.
    :return: (mean difference, 95% CI half-width, two-sided paired p-value).
        The last two are NaN with fewer than two usable pairs.
    """
    pairs = [
        (before, after)
        for base_trial, treat_trial in zip(baseline, treatment)
        if (before := extract(base_trial)) is not None
        and (after := extract(treat_trial)) is not None
    ]
    if not pairs:
        return float("nan"), float("nan"), float("nan")

    differences = [after - before for before, after in pairs]
    mean_difference = statistics.fmean(differences)
    if len(differences) < 2:
        return mean_difference, float("nan"), float("nan")

    half_width = (
        stats.t.ppf(0.975, len(differences) - 1)
        * statistics.stdev(differences)
        / len(differences) ** 0.5
    )
    p_value = float(stats.ttest_rel(
        [after for _, after in pairs],
        [before for before, _ in pairs],
    ).pvalue)
    return mean_difference, half_width, p_value


def served_pp(trial: TrialResult) -> float | None:
    """
    Served fraction as percentage points, or None when undefined.
    :param trial: One trial's results.
    :return: Served fraction x 100, or None.
    """
    if trial.kpi.served_fraction is None:
        return None
    return 100.0 * trial.kpi.served_fraction


def max_ride(trial: TrialResult) -> float | None:
    """
    Longest ride in one trial, or None when nobody was carried.
    :param trial: One trial's results.
    :return: Maximum ride time in seconds, or None.
    """
    return max(trial.ride_times) if trial.ride_times else None


def mean_of(
    trials: list[TrialResult],
    extract: Callable[[TrialResult], float | None],
) -> float:
    """
    Mean of one metric across trials, ignoring trials that lack it.
    :param trials: Per-trial results.
    :param extract: Maps a TrialResult to the scalar.
    :return: The mean, or NaN when no trial defined it.
    """
    values = [v for t in trials if (v := extract(t)) is not None]
    return statistics.fmean(values) if values else float("nan")


# (display name, extractor, direction that counts as an improvement)
_MetricSpec = tuple[str, Callable[[TrialResult], float | None], str]

METRICS: list[_MetricSpec] = [
    ("Served (pp)", served_pp, "higher"),
    ("Mean wait (s)", lambda r: r.kpi.mean_wait_time, "lower"),
    ("Mean ride (s)", lambda r: r.kpi.mean_ride_time, "lower"),
    ("Max ride (s)", max_ride, "lower"),
]

for metric_name, extract, better in METRICS:
    lines = [
        f"**{metric_name}**", "",
        "| Window | L0 | L1 | delta | p |",
        "| --- | --- | --- | --- | --- |",
    ]
    for window_name, _ in WINDOWS:
        l0_trials = summary.results[column(window_name, "L0")]
        l1_trials = summary.results[column(window_name, "L1")]
        delta, half_width, p_value = paired_delta(
            l0_trials, l1_trials, extract
        )
        improved = delta > 0 if better == "higher" else delta < 0
        mark = "**" if p_value < 0.05 and improved else ""
        lines.append(
            f"| {window_name} | {mean_of(l0_trials, extract):.1f} "
            f"| {mean_of(l1_trials, extract):.1f} "
            f"| {mark}{delta:+.1f} ± {half_width:.1f}{mark} "
            f"| {p_value:.4f} |"
        )
    display(Markdown("\n".join(lines)))

## 6. Does the gain track how bad the order was? (Q1)

The decisive plot for Q1: L1's ride improvement against L0's ride time at the
same window. A rising relationship means reordering is worth most exactly
where the policy is weakest.

In [ ]:
# Colourblind-safe pair in fixed order, validated for CVD separation.
MEAN_COLOUR = "#1f77b4"
MAX_COLOUR = "#e69f00"

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5))

for ax, (title, extract, colour) in zip(
    (left, right),
    [("Mean ride", lambda r: r.kpi.mean_ride_time, MEAN_COLOUR),
     ("Max ride", max_ride, MAX_COLOUR)],
):
    baselines, improvements = [], []
    for window_name, _ in WINDOWS:
        l0_trials = summary.results[column(window_name, "L0")]
        l1_trials = summary.results[column(window_name, "L1")]
        baseline = mean_of(l0_trials, extract)
        delta, _, _ = paired_delta(l0_trials, l1_trials, extract)
        baselines.append(baseline)
        # Plotted as a saving, so "up" means "L1 helped more".
        improvements.append(-delta)
        ax.annotate(window_name, xy=(baseline, -delta), xytext=(6, 6),
                    textcoords="offset points", fontsize=9, color="0.3")
    ax.plot(baselines, improvements, "o", color=colour, markersize=10)
    ax.axhline(0, color="0.5", linewidth=1)
    ax.set_xlabel(f"L0 {title.lower()} (s)")
    ax.set_ylabel(f"Seconds saved by L1")
    ax.set_title(f"{title}: does the saving grow with the problem?")
    ax.grid(True, alpha=0.3)
    ax.margins(0.15)

fig.suptitle("Q1: reordering should pay most where the order is worst")
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
plt.show()

## 7. Refusals and repair effort (Q2, Q4)

Whether the slack channel ever opens up, and how much work each acceptance
costs.

In [ ]:
def totals(label: str) -> dict[str, float]:
    """
    Sum the counters of every policy built for one column.
    :param label: Column label.
    :return: Counter name to run total, empty if the column never ran.
    """
    merged: dict[str, float] = defaultdict(float)
    for policy in INSTANCES[label]:
        for key, value in policy.stats().items():
            merged[key] += value
    return dict(merged)


def rate(numerator: float, denominator: float) -> float:
    """
    Percentage ratio degrading to NaN rather than raising.
    :param numerator: Count of events.
    :param denominator: Count of opportunities.
    :return: Percentage, or NaN when denominator is zero.
    """
    if not denominator:
        return float("nan")
    return 100.0 * numerator / denominator


def rejection_rate(totals_of_column: dict[str, float]) -> str:
    """
    Format one column's first-pass rejection rate.
    :param totals_of_column: Run totals for one column.
    :return: The formatted percentage.
    """
    value = rate(
        totals_of_column.get("rejections", 0.0),
        totals_of_column.get("insertion_attempts", 0.0),
    )
    return f"{value:.1f} %"


def unscheduled_rate(totals_of_column: dict[str, float]) -> str:
    """
    Format one column's share of waiting requests left unscheduled.
    :param totals_of_column: Run totals for one column.
    :return: The formatted percentage.
    """
    value = rate(
        totals_of_column.get("left_unscheduled", 0.0),
        totals_of_column.get("offered", 0.0),
    )
    return f"{value:.1f} %"


TOTALS = {label: totals(label) for label in COLUMNS}

lines = [
    "| Window | L0 unscheduled | L1 unscheduled | L0 rejections "
    "| L1 rejections | relocations accepted | acceptance rate |",
    "| --- | --- | --- | --- | --- | --- | --- |",
]
for window_name, _ in WINDOWS:
    l0 = TOTALS[column(window_name, "L0")]
    l1 = TOTALS[column(window_name, "L1")]
    accepted = l1.get("pair_accepted", 0.0) + l1.get("dropoff_accepted", 0.0)
    tried = l1.get("pair_attempts", 0.0) + l1.get("dropoff_attempts", 0.0)
    lines.append(
        f"| {window_name} "
        f"| {unscheduled_rate(l0)} | {unscheduled_rate(l1)} "
        f"| {rejection_rate(l0)} | {rejection_rate(l1)} "
        f"| {accepted:,.0f} | {rate(accepted, tried):.1f} % |"
    )
display(Markdown("\n".join(lines)))
display(Markdown(
    "*Unscheduled and rejection columns are the two refusal measures from "
    "the L1 ablation: the first depends on no internal call structure and "
    "wins any disagreement.*"
))

## 8. The frontier, before and after (Q3)

Served fraction against the two ride metrics. L1 should pull every point
leftward; the question is whether any of them reaches the fixed-route cap.

In [ ]:
# Fixed-route reference values from baseline-c0.ipynb (800 trials,
# commitment=0). Drawn as lines because they are what the demand-responsive
# family has to beat.
FIXED_MAX_RIDE = 1302.0
FIXED_MEAN_RIDE = 654.0
FIXED_SERVED = 68.6

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5))
for ax, (title, extract, reference) in zip(
    (left, right),
    [("Mean ride", lambda r: r.kpi.mean_ride_time, FIXED_MEAN_RIDE),
     ("Max ride", max_ride, FIXED_MAX_RIDE)],
):
    for level, colour, marker in [("L0", "#1f77b4", "o"),
                                  ("L1", "#009e73", "s")]:
        xs, ys = [], []
        for window_name, _ in WINDOWS:
            trials = summary.results[column(window_name, level)]
            xs.append(mean_of(trials, extract))
            ys.append(mean_of(trials, served_pp))
        ax.plot(xs, ys, marker, color=colour, markersize=9,
                linestyle="-", linewidth=1.5, label=level)
        if level == "L1":
            for window_name, x, y in zip(
                [name for name, _ in WINDOWS], xs, ys
            ):
                ax.annotate(window_name, xy=(x, y), xytext=(6, -12),
                            textcoords="offset points", fontsize=9,
                            color="0.3")
    ax.axvline(reference, color="0.35", linestyle="--", linewidth=1.4)
    ax.annotate("fixed route", xy=(reference, FIXED_SERVED),
                xytext=(6, 0), textcoords="offset points",
                fontsize=9, color="0.35")
    ax.axhline(80.0, color="0.7", linestyle=":", linewidth=1.2)
    ax.set_xlabel(f"{title} (s)  — lower is better")
    ax.set_ylabel("Served fraction (%)")
    ax.set_title(f"Coverage against {title.lower()}")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("Q3: does reordering move the frontier? "
             "(dotted line = the 80% coverage floor)")
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
plt.show()
display(Markdown(summary.markdown))

## What to look for

**Regression check first.** Every L0 column should reproduce its counterpart in
`baseline-c0.ipynb` — no-window 82.6% served / 356 s mean ride, W450 82.8% /
874 s, W900 93.1% / 521 s. That run used 800 trials to this one's 400, so
expect wider intervals rather than different means. A disagreement in the
means matters more than anything else on the page.

**Q1 — the scaling plot.** The mechanism says reordering fixes bad orders, so
the saving should grow with how bad the order was.

- *Rising* → confirmed, and L1 is worth most exactly where the policy is
  weakest. The tight windows become far more usable than the baseline suggests.
- *Flat* → the ~12% from W900 is simply what reordering is worth, everywhere.
  Still a good result, just not a compounding one.
- *Falling* → **refutes the ride-quality mechanism.** If repair cannot fix the
  worst orders, then W450's 4756 s ride tail is not an ordering artefact but
  the deadline forcing a genuinely bad route, and no amount of L2 would help.
  That would also mean the W900 result needs re-explaining.

**Q2 — refusals.** At W900 they barely moved. Tight windows refuse far more, so
this is where a slack effect would show up if it exists at all. Flat
everywhere confirms ride quality is the whole story; a drop at W450/W600 means
the slack channel is real but only when the window binds — which would matter
for designing L2.

**Q3 — the frontier plots.** The specific thing to look for is any L1 point
that gets left of the fixed-route line while staying above the 80% floor.
None currently does on max ride, and closing a 1300 s gap is a lot to ask of
an 11% improvement. Mean ride is the more winnable one — the no-window
ablation is already well left of fixed-route there.

**Q4 — the unbounded column.** If L1 improves the no-window ablation too,
reordering is a general win rather than compensation for a badly-behaved
window. If it does nothing there, reordering is specifically a *deadline*
repair, which is a narrower and more interesting claim.

**Also worth reading off:** the acceptance-rate column. It was 2.5% at W900,
which is where L1's 3.4× cost goes. If tight windows accept a much higher
fraction, a cheap filter that skips hopeless relocations would pay for itself
quickly.

### A caveat on reproducibility

Set `PYTHONHASHSEED=0` before launching Jupyter.

### Follow-ups this could trigger

```python
# Do extra repair sweeps compound? Each accepted move changes the schedule
# the next candidate is judged against, so a second pass may find more.
POLICIES = [
    (f"passes={n}", make_factory(f"passes={n}", CountingL1, 900.0))
    for n in (1, 2, 3)
]

# Does the win survive heavier demand?
summary_heavy = summarize_comparison(
    policies=POLICIES, graph=graph, n_trials=200, base_seed=0,
    max_requests=500, capacity=40, max_wait_s=900.0, commitment=0,
)
```